In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/compas_preprocessed.npz"
OUT_DIR = "./outputs_compas"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 10
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")

    print(f"Loading data from {npz_path}...")
    d = np.load(npz_path, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test  = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test  = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Shapes -> X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape}")
    return X_train, y_train, X_test, y_test, feature_names

X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
n_features = X_train.shape[1]
all_labels = sorted(np.unique(y_train).tolist())
baseline_full = np.mean(X_train, axis=0)

trained_models = {}
MODEL_NAMES = ["logistic", "svm", "random_forest"]

print("\nTraining models...")
for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[j]) for j in feature_subset]
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=subset_names
        )

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

records = []
print(f"\nRunning explanation on {len(test_idx)} samples...")

for model_name, model in trained_models.items():
    for i, idx in enumerate(test_idx):
        x = X_test[idx]
        y = y_test[idx]
        feat_subset = np.arange(n_features)

        util = LocalLossUtility(
            model=model, x_full=x, y_true=y, baseline_full=baseline_full,
            feature_subset=feat_subset, feature_names=feature_names, all_labels=all_labels
        )

        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                records.append({
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y),
                    "method": method_name,
                    "alloc_full": alloc.tolist()
                })
            except Exception as e:
                print(f"Error in {method_name}: {e}")

df = pd.DataFrame(records)
# Save the results indicating the Loss Utility usage for COMPAS
df.to_csv(os.path.join(OUT_DIR, "loss_utility_results.csv"), index=False)
print("Done.")

Loading data from ../../datasets/compas_preprocessed.npz...
Shapes -> X_train: (4222, 19), y_train: (4222,), X_test: (1056, 19)

Training models...

Running explanation on 10 samples...


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/compas_preprocessed.npz"
OUT_DIR = "./outputs_compas"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 10
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test  = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test  = d["y_test"].astype(int).ravel()
    feature_names = d["feature_names"].tolist() if "feature_names" in d else [f"f{i}" for i in range(X_train.shape[1])]
    return X_train, y_train, X_test, y_test, feature_names

X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
all_labels = sorted(np.unique(y_train).tolist())
baseline_full = np.mean(X_train, axis=0)

trained_models = {}
for m_name in ["logistic", "svm", "random_forest"]:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

def get_compas_groups(feat_names):
    group_dict = {
        'age': [], 'age_cat': [], 'juvenile_crimes': [], 'priors_count': [],
        'c_charge_degree': [], 'sex': [], 'race': []
    }
    for i, name in enumerate(feat_names):
        if name == 'age': group_dict['age'].append(i)
        elif 'age_cat' in name: group_dict['age_cat'].append(i)
        elif 'juv' in name: group_dict['juvenile_crimes'].append(i)
        elif 'priors' in name: group_dict['priors_count'].append(i)
        elif 'charge' in name: group_dict['c_charge_degree'].append(i)
        elif 'sex' in name: group_dict['sex'].append(i)
        elif 'race' in name: group_dict['race'].append(i)
    return list(group_dict.keys()), list(group_dict.values())

group_names, group_indices = get_compas_groups(feature_names)
print(f"\nGrouped 19 columns into EXACTLY {len(group_names)} base features: {group_names}")

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class GroupedLocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, group_names, group_indices, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.group_indices = group_indices
        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.data = FeaturePlayers(indices=list(range(len(group_names))), data_names=group_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for group_idx in S:
            for col_idx in self.group_indices[int(group_idx)]:
                x[col_idx] = self.x_full[col_idx]
        return x.reshape(1, -1)

    def __call__(self, S):
        x_in = self._x_for_coalition(S)
        if hasattr(self.model, "predict_proba"):
            return -float(log_loss([self.y_true], self.model.predict_proba(x_in), labels=self.all_labels))
        return 0.0 if self.model.predict(x_in)[0] == self.y_true else -1.0

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value), ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus), ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index)
]

records = []
print(f"\nRunning explanation on {len(test_idx)} samples...")

for model_name, model in trained_models.items():
    for i, idx in enumerate(test_idx):
        util = GroupedLocalLossUtility(model, X_test[idx], y_test[idx], baseline_full, group_names, group_indices, all_labels)
        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                records.append({
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y_test[idx]),
                    "method": method_name,
                    "alloc_full": alloc.tolist(), # Stores exactly 7 values
                    "alloc_named": {group_names[j]: float(alloc[j]) for j in range(len(group_names))}
                })
            except Exception as e:
                print(f"Error in {method_name}: {e}")

df = pd.DataFrame(records)
out_path = os.path.join(OUT_DIR, "loss_utility_results.csv")
df.to_csv(out_path, index=False)
print(f"Done. Results saved to {out_path}")


Grouped 19 columns into EXACTLY 7 base features: ['age', 'age_cat', 'juvenile_crimes', 'priors_count', 'c_charge_degree', 'sex', 'race']

Running explanation on 10 samples...
Done. Results saved to ./outputs_compas/loss_utility_results.csv


In [2]:
import os
import sys
import warnings
import logging
import ast
from pathlib import Path
from dataclasses import dataclass
from typing import List, Iterator

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus
from sklearn.preprocessing import LabelEncoder

# Setup Paths
sys.path.append(os.path.abspath(".."))

from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index
from models.train_model import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
DATA_NPZ_PATH = "../../datasets/compas_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_compas/loss_utility_results.csv"
OUT_DIR = Path("./quantus_compas/quantus_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV_PATH = OUT_DIR / "quantus_results_incremental.csv"

RANDOM_STATE = 42
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

# Utilities & grouping
def get_compas_groups(feat_names):
    group_dict = {
        'age': [], 'age_cat': [], 'juvenile_crimes': [], 'priors_count': [],
        'c_charge_degree': [], 'sex': [], 'race': []
    }
    for i, name in enumerate(feat_names):
        if name == 'age': group_dict['age'].append(i)
        elif 'age_cat' in name: group_dict['age_cat'].append(i)
        elif 'juv' in name: group_dict['juvenile_crimes'].append(i)
        elif 'priors' in name: group_dict['priors_count'].append(i)
        elif 'charge' in name: group_dict['c_charge_degree'].append(i)
        elif 'sex' in name: group_dict['sex'].append(i)
        elif 'race' in name: group_dict['race'].append(i)
    return list(group_dict.keys()), list(group_dict.values())

@dataclass
class FeaturePlayers:
    indices: List[int]; data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class GroupedLocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, group_names, group_indices, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.group_indices = group_indices
        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.data = FeaturePlayers(indices=list(range(len(group_names))), data_names=group_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for group_idx in S:
            for col_idx in self.group_indices[int(group_idx)]:
                x[col_idx] = self.x_full[col_idx]
        return x.reshape(1, -1)

    def __call__(self, S):
        x_in = self._x_for_coalition(S)
        if hasattr(self.model, "predict_proba"):
            return -float(log_loss([self.y_true], self.model.predict_proba(x_in), labels=self.all_labels))
        return 0.0 if self.model.predict(x_in)[0] == self.y_true else -1.0

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        x_np = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else x
        if x_np.ndim == 3: x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1: x_np = x_np.reshape(1, -1)
        out = self.model.predict_proba(x_np) if self.is_clf else self.model.predict(x_np)
        if not self.is_clf and out.ndim == 1: out = out.reshape(-1, 1)
        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train, X_test = d["X_train"].astype(float), d["X_test"].astype(float)
    y_train, y_test = d["y_train"].astype(int).ravel(), d["y_test"].astype(int).ravel()
    feat_names = d["feature_names"].tolist() if "feature_names" in d else [f"f{i}" for i in range(X_train.shape[1])]
    return X_train, X_test, y_train, y_test, feat_names

# Metrics & perturbation
def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()
    F = arr.shape[1] if arr.ndim == 2 else arr.shape[2]
    for b in range(arr.shape[0]):
        idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
        valid_idxs = [i for i in idx_list if i < F]
        if arr.ndim == 2: arr_pert[b][valid_idxs] = base_vec[valid_idxs]
        else: arr_pert[b, 0][valid_idxs] = base_vec[valid_idxs]
    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    noise = np.random.uniform(0.01, 0.02, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b): return np.linalg.norm(np.array(a) - np.array(b))

def get_metrics(baseline_vec):
    kwargs = {"baseline": baseline_vec.reshape(1, 1, -1)}
    return {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(nr_runs=10, subset_size=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "selectivity": quantus.Selectivity(patch_size=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "monotonicity": quantus.Monotonicity(features_in_step=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "pixel_flipping": quantus.PixelFlipping(features_in_step=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),
        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(nr_samples=1, perturb_std=0.1, norm_numerator=distance_euclidean, norm_denominator=distance_euclidean, perturb_func=perturb_uniform_robust, perturb_func_kwargs={}, return_aggregate=True),
        "max_sensitivity": quantus.MaxSensitivity(nr_samples=1, lower_bound=0.1, upper_bound=0.2, norm_numerator=quantus.norm_func.fro_norm, norm_denominator=quantus.norm_func.fro_norm, perturb_func=perturb_uniform_robust, perturb_func_kwargs={}, return_aggregate=True)
    }

def compute_aopc_manual(sk_model, X_test_2d, y_test_indices, attributions_2d, top_k_list, baseline):
    if not hasattr(sk_model, "predict_proba"): return np.nan
    base_scores = sk_model.predict_proba(X_test_2d)[np.arange(len(y_test_indices)), y_test_indices]
    drops = []
    for k in [k for k in top_k_list if k <= X_test_2d.shape[1]]:
        X_pert = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_idx = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_pert[i, top_idx] = baseline[top_idx]
        new_scores = sk_model.predict_proba(X_pert)[np.arange(len(y_test_indices)), y_test_indices]
        drops.append(np.mean(base_scores - new_scores))
    return np.mean(drops)

# Robust explainer wrapper for quantus
def make_explain_func(method_name, all_labels, baseline_full, group_names, group_indices):
    METHOD_MAP = {
        "shapley": exact_shapley_value, "least_core": exact_least_core,
        "nucleolus": exact_nucleolus, "gately": exact_gately_value, "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        real_model = model.model if isinstance(model, SklearnTorchWrapper) else model
        attributions = []
        for i in range(len(inputs)):
            x = inputs[i].detach().cpu().numpy().flatten() if hasattr(inputs[i], "detach") else inputs[i].flatten()
            y = int(targets[i].item()) if hasattr(targets[i], "item") else int(targets[i])
            real_label = real_model.classes_[y] if hasattr(real_model, "classes_") else y

            util = GroupedLocalLossUtility(real_model, x, real_label, baseline_full, group_names, group_indices, all_labels)

            expanded = np.zeros(len(x))
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                # Map the 7 features back onto the 19 raw columns for Quantus
                for g_idx, col_indices in enumerate(group_indices):
                    for col_idx in col_indices:
                        expanded[col_idx] = alloc[g_idx] / len(col_indices)
            except Exception: pass
            attributions.append(expanded.reshape(1, -1))
        return np.array(attributions)
    return explain_func

# Main
def main():
    X_train, X_test, y_train, y_test, feature_names = load_data_strict()
    baseline = np.mean(X_train, axis=0)
    all_labels = sorted(np.unique(y_train).tolist())
    group_names, group_indices = get_compas_groups(feature_names)

    le = LabelEncoder()
    le.fit(y_train)

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset_indices = le.transform(y_test[valid_indices])
    x_batch = X_subset[:, np.newaxis, :]

    cols = ["model", "method", "faithfulness_corr", "faithfulness_estimate", "selectivity", "monotonicity", "pixel_flipping", "sparseness", "complexity", "local_lipschitz_estimate", "max_sensitivity", "aopc_manual"]
    if not OUT_CSV_PATH.exists():
        pd.DataFrame(columns=cols).to_csv(OUT_CSV_PATH, index=False)
        processed_pairs = set()
    else:
        existing_df = pd.read_csv(OUT_CSV_PATH)
        processed_pairs = set(zip(existing_df["model"], existing_df["method"]))

    metrics = get_metrics(baseline)

    for model_name in df_results["model"].unique():
        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()
        df_model = df_results[df_results["model"] == model_name]

        for method_name in df_results["method"].unique():
            if (model_name, method_name) in processed_pairs: continue
            logger.info(f"Evaluating {model_name} - {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                val_arr_19 = np.zeros(X_train.shape[1])
                if not row.empty:
                    val_str = row.iloc[0]["alloc_full"]
                    val_arr_grouped = np.array(ast.literal_eval(val_str)) if isinstance(val_str, str) else np.array(val_str)

                    # Distribute the 7 calculated attributions across the 19 original columns
                    for g_idx, col_indices in enumerate(group_indices):
                        for col_idx in col_indices:
                            val_arr_19[col_idx] = val_arr_grouped[g_idx] / len(col_indices)

                attributions_list.append(val_arr_19)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]
            row_stats = {"model": model_name, "method": method_name}

            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try: row_stats[m_name] = np.nanmean(metric_fn(model=torch_model, x_batch=x_batch, y_batch=y_subset_indices, a_batch=a_batch_3d, channel_first=True, softmax=False))
                except Exception: row_stats[m_name] = np.nan

            try: row_stats["aopc_manual"] = compute_aopc_manual(sk_model, X_subset, y_subset_indices, a_batch_2d, AOPC_TOPK_LIST, baseline)
            except Exception: row_stats["aopc_manual"] = np.nan

            explain_func = make_explain_func(method_name, all_labels, baseline, group_names, group_indices)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                if model_name == "random_forest":
                     row_stats[m_name] = np.nan
                     continue
                try: row_stats[m_name] = np.nanmean(metrics[m_name](model=torch_model, x_batch=x_batch, y_batch=y_subset_indices, explain_func=explain_func, channel_first=True, softmax=False))
                except Exception: row_stats[m_name] = np.nan

            df_row = pd.DataFrame([row_stats]).reindex(columns=cols)
            df_row.to_csv(OUT_CSV_PATH, mode='a', header=False, index=False)

if __name__ == "__main__":
    main()

In [4]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/compas_preprocessed.npz"
OUT_DIR = "./outputs_compas"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test  = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test  = d["y_test"].astype(int).ravel()
    feature_names = d["feature_names"].tolist() if "feature_names" in d else [f"f{i}" for i in range(X_train.shape[1])]
    return X_train, y_train, X_test, y_test, feature_names

X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
all_labels = sorted(np.unique(y_train).tolist())
baseline_full = np.mean(X_train, axis=0)

trained_models = {}
for m_name in ["logistic", "svm", "random_forest"]:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

def get_compas_groups(feat_names):
    group_dict = {
        'age': [], 'age_cat': [], 'juvenile_crimes': [], 'priors_count': [],
        'c_charge_degree': [], 'sex': [], 'race': []
    }
    for i, name in enumerate(feat_names):
        if name == 'age': group_dict['age'].append(i)
        elif 'age_cat' in name: group_dict['age_cat'].append(i)
        elif 'juv' in name: group_dict['juvenile_crimes'].append(i)
        elif 'priors' in name: group_dict['priors_count'].append(i)
        elif 'charge' in name: group_dict['c_charge_degree'].append(i)
        elif 'sex' in name: group_dict['sex'].append(i)
        elif 'race' in name: group_dict['race'].append(i)
    return list(group_dict.keys()), list(group_dict.values())

group_names, group_indices = get_compas_groups(feature_names)
print(f"\nGrouped 19 columns into EXACTLY {len(group_names)} base features: {group_names}")

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class GroupedLocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, group_names, group_indices, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.group_indices = group_indices
        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.data = FeaturePlayers(indices=list(range(len(group_names))), data_names=group_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for group_idx in S:
            for col_idx in self.group_indices[int(group_idx)]:
                x[col_idx] = self.x_full[col_idx]
        return x.reshape(1, -1)

    def __call__(self, S):
        x_in = self._x_for_coalition(S)
        if hasattr(self.model, "predict_proba"):
            return -float(log_loss([self.y_true], self.model.predict_proba(x_in), labels=self.all_labels))
        return 0.0 if self.model.predict(x_in)[0] == self.y_true else -1.0

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value), ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus), ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index)
]

records = []
print(f"\nRunning explanation on {len(test_idx)} samples...")

for model_name, model in trained_models.items():
    for i, idx in enumerate(test_idx):
        util = GroupedLocalLossUtility(model, X_test[idx], y_test[idx], baseline_full, group_names, group_indices, all_labels)
        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                records.append({
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y_test[idx]),
                    "method": method_name,
                    "alloc_full": alloc.tolist(), # Stores exactly 7 values
                    "alloc_named": {group_names[j]: float(alloc[j]) for j in range(len(group_names))}
                })
            except Exception as e:
                print(f"Error in {method_name}: {e}")

df = pd.DataFrame(records)
out_path = os.path.join(OUT_DIR, "loss_utility_results.csv")
df.to_csv(out_path, index=False)
print(f"Done. Results saved to {out_path}")


Grouped 19 columns into EXACTLY 7 base features: ['age', 'age_cat', 'juvenile_crimes', 'priors_count', 'c_charge_degree', 'sex', 'race']

Running explanation on 50 samples...
Done. Results saved to ./outputs_compas/loss_utility_results.csv


In [8]:
import os
import sys
import warnings
import logging
import ast
from pathlib import Path
from dataclasses import dataclass
from typing import List, Iterator

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus
from sklearn.preprocessing import LabelEncoder

# Setup Paths
sys.path.append(os.path.abspath(".."))

from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index
from models.train_model import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
DATA_NPZ_PATH = "../../datasets/compas_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_compas/loss_utility_results.csv"
OUT_DIR = Path("./quantus_compas/quantus_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV_PATH = OUT_DIR / "quantus_results_incremental.csv"

RANDOM_STATE = 42
# Exactly 7 logical features for COMPAS
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

# Utilities & grouping
def get_compas_groups(feat_names):
    group_dict = {
        'age': [], 'age_cat': [], 'juvenile_crimes': [], 'priors_count': [],
        'c_charge_degree': [], 'sex': [], 'race': []
    }
    for i, name in enumerate(feat_names):
        if name == 'age': group_dict['age'].append(i)
        elif 'age_cat' in name: group_dict['age_cat'].append(i)
        elif 'juv' in name: group_dict['juvenile_crimes'].append(i)
        elif 'priors' in name: group_dict['priors_count'].append(i)
        elif 'charge' in name: group_dict['c_charge_degree'].append(i)
        elif 'sex' in name: group_dict['sex'].append(i)
        elif 'race' in name: group_dict['race'].append(i)
    return list(group_dict.keys()), list(group_dict.values())

@dataclass
class FeaturePlayers:
    indices: List[int]; data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class GroupedLocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, group_names, group_indices, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.group_indices = group_indices
        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.data = FeaturePlayers(indices=list(range(len(group_names))), data_names=group_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for group_idx in S:
            for col_idx in self.group_indices[int(group_idx)]:
                x[col_idx] = self.x_full[col_idx]
        return x.reshape(1, -1)

    def __call__(self, S):
        x_in = self._x_for_coalition(S)
        if hasattr(self.model, "predict_proba"):
            return -float(log_loss([self.y_true], self.model.predict_proba(x_in), labels=self.all_labels))
        return 0.0 if self.model.predict(x_in)[0] == self.y_true else -1.0

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        x_np = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else x
        if x_np.ndim == 3: x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1: x_np = x_np.reshape(1, -1)
        out = self.model.predict_proba(x_np) if self.is_clf else self.model.predict(x_np)
        if not self.is_clf and out.ndim == 1: out = out.reshape(-1, 1)
        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train, X_test = d["X_train"].astype(float), d["X_test"].astype(float)
    y_train, y_test = d["y_train"].astype(int).ravel(), d["y_test"].astype(int).ravel()
    feat_names = d["feature_names"].tolist() if "feature_names" in d else [f"f{i}" for i in range(X_train.shape[1])]
    return X_train, X_test, y_train, y_test, feat_names

# Metrics & perturbation
def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()
    F = arr.shape[1] if arr.ndim == 2 else arr.shape[2]
    for b in range(arr.shape[0]):
        idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
        valid_idxs = [i for i in idx_list if i < F]
        if arr.ndim == 2: arr_pert[b][valid_idxs] = base_vec[valid_idxs]
        else: arr_pert[b, 0][valid_idxs] = base_vec[valid_idxs]
    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    noise = np.random.uniform(0.01, 0.02, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b): return np.linalg.norm(np.array(a) - np.array(b))

def get_metrics(baseline_vec):
    kwargs = {"baseline": baseline_vec.reshape(1, 1, -1)}
    return {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(nr_runs=10, subset_size=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "selectivity": quantus.Selectivity(patch_size=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "monotonicity": quantus.Monotonicity(features_in_step=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "pixel_flipping": quantus.PixelFlipping(features_in_step=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),
        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(nr_samples=1, perturb_std=0.1, norm_numerator=distance_euclidean, norm_denominator=distance_euclidean, perturb_func=perturb_uniform_robust, perturb_func_kwargs={}, return_aggregate=True),
        "max_sensitivity": quantus.MaxSensitivity(nr_samples=1, lower_bound=0.1, upper_bound=0.2, norm_numerator=quantus.norm_func.fro_norm, norm_denominator=quantus.norm_func.fro_norm, perturb_func=perturb_uniform_robust, perturb_func_kwargs={}, return_aggregate=True)
    }

def compute_aopc_manual(sk_model, X_test_2d, y_test_indices, attributions_grouped, top_k_list, baseline, group_indices):
    """
    Evaluates AOPC by dropping entire groups of features based on the top grouped attributions.
    """
    if not hasattr(sk_model, "predict_proba"):
        return np.nan

    base_probs = sk_model.predict_proba(X_test_2d)
    base_scores = base_probs[np.arange(len(y_test_indices)), y_test_indices]

    drops_all_k = []
    # Ensure we don't try to drop more groups than exist (7)
    valid_k_list = [k for k in top_k_list if k <= len(group_indices)]

    for k in valid_k_list:
        X_perturbed = X_test_2d.copy()

        for i in range(len(X_test_2d)):
            # Find the top K *groups* based on the 7 attributions
            top_groups = np.argsort(-np.abs(attributions_grouped[i]))[:k]

            # Drop all one-hot columns associated with those top K groups
            for g_idx in top_groups:
                cols_to_drop = group_indices[g_idx]
                X_perturbed[i, cols_to_drop] = baseline[cols_to_drop]

        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test_indices)), y_test_indices]
        drops_all_k.append(np.mean(base_scores - new_scores))

    return np.mean(drops_all_k) if drops_all_k else np.nan

# Robust explainer wrapper for quantus
def make_explain_func(method_name, all_labels, baseline_full, group_names, group_indices):
    METHOD_MAP = {
        "shapley": exact_shapley_value, "least_core": exact_least_core,
        "nucleolus": exact_nucleolus, "gately": exact_gately_value, "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        real_model = model.model if isinstance(model, SklearnTorchWrapper) else model
        attributions = []
        for i in range(len(inputs)):
            x = inputs[i].detach().cpu().numpy().flatten() if hasattr(inputs[i], "detach") else inputs[i].flatten()
            y = int(targets[i].item()) if hasattr(targets[i], "item") else int(targets[i])
            real_label = real_model.classes_[y] if hasattr(real_model, "classes_") else y

            util = GroupedLocalLossUtility(real_model, x, real_label, baseline_full, group_names, group_indices, all_labels)

            expanded = np.zeros(len(x))
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                # Map the 7 features back onto the 19 raw columns for Quantus
                for g_idx, col_indices in enumerate(group_indices):
                    for col_idx in col_indices:
                        expanded[col_idx] = alloc[g_idx] / len(col_indices)
            except Exception: pass
            attributions.append(expanded.reshape(1, -1))
        return np.array(attributions)
    return explain_func

# Main
def main():
    if not os.path.exists(RESULTS_CSV_PATH):
        print(f"ERROR: {RESULTS_CSV_PATH} not found. You must run the Explainer script first!")
        return

    X_train, X_test, y_train, y_test, feature_names = load_data_strict()
    baseline = np.mean(X_train, axis=0)
    all_labels = sorted(np.unique(y_train).tolist())
    group_names, group_indices = get_compas_groups(feature_names)

    le = LabelEncoder()
    le.fit(y_train)

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset_indices = le.transform(y_test[valid_indices])
    x_batch = X_subset[:, np.newaxis, :]

    cols = ["model", "method", "faithfulness_corr", "faithfulness_estimate", "selectivity", "monotonicity", "pixel_flipping", "sparseness", "complexity", "local_lipschitz_estimate", "max_sensitivity", "aopc_manual"]
    if not OUT_CSV_PATH.exists():
        pd.DataFrame(columns=cols).to_csv(OUT_CSV_PATH, index=False)
        processed_pairs = set()
    else:
        existing_df = pd.read_csv(OUT_CSV_PATH)
        processed_pairs = set(zip(existing_df["model"], existing_df["method"]))
        print(f"Resuming... Found {len(processed_pairs)} already processed entries.")

    metrics = get_metrics(baseline)

    for model_name in df_results["model"].unique():
        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()
        df_model = df_results[df_results["model"] == model_name]

        for method_name in df_results["method"].unique():
            if (model_name, method_name) in processed_pairs:
                print(f"Skipping {model_name} - {method_name} (Already in CSV)")
                continue

            print(f"Evaluating {model_name} - {method_name}...")

            attributions_list = []
            attributions_grouped_list = [] # <--- Track grouped for AOPC

            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                val_arr_19 = np.zeros(X_train.shape[1])
                val_arr_grouped = np.zeros(len(group_indices))

                if not row.empty:
                    val_str = row.iloc[0]["alloc_full"]
                    val_arr_grouped = np.array(ast.literal_eval(val_str)) if isinstance(val_str, str) else np.array(val_str)

                    # Distribute the 7 calculated attributions across the 19 original columns
                    for g_idx, col_indices in enumerate(group_indices):
                        for col_idx in col_indices:
                            val_arr_19[col_idx] = val_arr_grouped[g_idx] / len(col_indices)

                attributions_list.append(val_arr_19)
                attributions_grouped_list.append(val_arr_grouped)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]
            a_grouped_2d = np.array(attributions_grouped_list) # <--- To be passed to AOPC

            row_stats = {"model": model_name, "method": method_name}

            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try: row_stats[m_name] = np.nanmean(metric_fn(model=torch_model, x_batch=x_batch, y_batch=y_subset_indices, a_batch=a_batch_3d, channel_first=True, softmax=False))
                except Exception: row_stats[m_name] = np.nan

            try:
                row_stats["aopc_manual"] = compute_aopc_manual(
                    sk_model, X_subset, y_subset_indices,
                    a_grouped_2d, AOPC_TOPK_LIST, baseline, group_indices
                )
            except Exception as e:
                row_stats["aopc_manual"] = np.nan

            explain_func = make_explain_func(method_name, all_labels, baseline, group_names, group_indices)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                if model_name == "random_forest":
                     row_stats[m_name] = np.nan
                     continue
                try: row_stats[m_name] = np.nanmean(metrics[m_name](model=torch_model, x_batch=x_batch, y_batch=y_subset_indices, explain_func=explain_func, channel_first=True, softmax=False))
                except Exception: row_stats[m_name] = np.nan

            df_row = pd.DataFrame([row_stats]).reindex(columns=cols)
            df_row.to_csv(OUT_CSV_PATH, mode='a', header=False, index=False)
            print(f"Saved results for {model_name} - {method_name}")

    print("Finished evaluating all models and methods!")

if __name__ == "__main__":
    main()

Resuming... Found 0 already processed entries.
Evaluating logistic - shapley...
Saved results for logistic - shapley
Evaluating logistic - least_core...
Saved results for logistic - least_core
Evaluating logistic - nucleolus...
Saved results for logistic - nucleolus
Evaluating logistic - gately...
Saved results for logistic - gately
Evaluating logistic - banzhaf...
Saved results for logistic - banzhaf
Evaluating svm - shapley...
Saved results for svm - shapley
Evaluating svm - least_core...
Saved results for svm - least_core
Evaluating svm - nucleolus...
Saved results for svm - nucleolus
Evaluating svm - gately...
Saved results for svm - gately
Evaluating svm - banzhaf...
Saved results for svm - banzhaf
Evaluating random_forest - shapley...
Saved results for random_forest - shapley
Evaluating random_forest - least_core...
Saved results for random_forest - least_core
Evaluating random_forest - nucleolus...
Saved results for random_forest - nucleolus
Evaluating random_forest - gately...
